# 01 描述性分析與規則回測
研究問題是歷史行程能否提供目的地捷徑與常用時段。此檔為描述性分析、規則與回測，不訓練 ML。
描述統計可用全期間；回測輸入須限於預測當時已知歷史。真實資料與逐筆結果未包含於公開版本；作者同意公開的彙總命中率列於文末。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
csv_path = Path('../data/yoxi_cleaned.csv')
if not csv_path.is_file():
    raise FileNotFoundError(f'找不到清理檔：{csv_path}；請將已授權的清理檔放在專案 data/ 目錄。')
required = ['Id', 'RiderId', 'OrderFormCreatedTime_UTC8', 'OrderStartDateTime_UTC8', 'PickUpLatitude',
    'PickUpLongitude', 'DropOffLatitude', 'DropOffLongitude', 'TravelDistance']
trips = pd.read_csv(csv_path, encoding='utf-8-sig', dtype={'Id': 'string', 'RiderId': 'string'})
missing_columns = sorted(set(required) - set(trips.columns))
if missing_columns:
    raise ValueError(f'缺少必要欄位：{missing_columns}')
for col in ('OrderFormCreatedTime_UTC8', 'OrderStartDateTime_UTC8'):
    trips[col] = pd.to_datetime(trips[col], errors='coerce')
required_na = trips[required].isna().sum()
if required_na.any():
    raise ValueError(f'必要欄位解析後缺值，請先核查清理檔：{required_na[required_na.gt(0)].to_dict()}')
if trips['Id'].duplicated().any():
    raise ValueError('Id 有重複，與上一階段清理結果不符；請先核對檔案。')
print(f'讀入 {len(trips):,} 筆；{len(trips.columns)} 欄；訂單建立時間：{trips.OrderFormCreatedTime_UTC8.min()} ～ {trips.OrderFormCreatedTime_UTC8.max()}')


## 網格與描述統計
經緯度按每度 111,320 公尺換算，經度乘 cos(24.5°)，各軸除以 500 後向下取整。這是固定近似 500 公尺網格；實際尺度隨緯度變化，相鄰網格的點可能很近。
熱區、時段、路線與個人偏好使用全期間資料。路線僅表示上下車網格組合，不能直接判定住家、公司、通勤或機場。

In [ ]:
CELL_M = 500
LAT_METERS_PER_DEG = 111320.0
LON_METERS_PER_DEG = 111320.0 * np.cos(np.deg2rad(24.5))

def make_cell(lat, lon, cell_m=CELL_M):
    x = np.floor(pd.to_numeric(lon, errors='coerce') * LON_METERS_PER_DEG / cell_m)
    y = np.floor(pd.to_numeric(lat, errors='coerce') * LAT_METERS_PER_DEG / cell_m)
    return 'g' + str(cell_m) + '_' + x.astype('Int64').astype('string') + '_' + y.astype('Int64').astype('string')
trips['pickup_area'] = make_cell(trips['PickUpLatitude'], trips['PickUpLongitude'])
trips['dropoff_area'] = make_cell(trips['DropOffLatitude'], trips['DropOffLongitude'])
trips['created_hour'] = trips['OrderFormCreatedTime_UTC8'].dt.hour
trips['created_weekday'] = trips['OrderFormCreatedTime_UTC8'].dt.dayofweek
trips['created_week'] = trips['OrderFormCreatedTime_UTC8'].dt.to_period('W').astype('string')
print('上車區域數：', trips['pickup_area'].nunique(), '；下車區域數：', trips['dropoff_area'].nunique())


In [ ]:
pickup_hotspots = trips.groupby('pickup_area', observed=True).agg(rides=('Id', 'size'), riders=('RiderId',
    'nunique'), lat=('PickUpLatitude', 'median'), lon=('PickUpLongitude', 'median')).sort_values('rides',
    ascending=False)
hourly = trips.groupby(['created_weekday', 'created_hour'], observed=True).agg(rides=('Id', 'size'),
    riders=('RiderId', 'nunique')).sort_values('rides', ascending=False)
popular_routes = trips.groupby(['pickup_area', 'dropoff_area'], observed=True).agg(rides=('Id', 'size'),
    riders=('RiderId', 'nunique'), pickup_lat=('PickUpLatitude', 'median'), pickup_lon=('PickUpLongitude',
    'median'), dropoff_lat=('DropOffLatitude', 'median'), dropoff_lon=('DropOffLongitude',
    'median')).sort_values('rides', ascending=False)


個人路線候選需同一帳號、上車網格、下車網格至少 3 趟且跨至少 2 週，避免單日集中搭乘被視為固定習慣。比例分母是該帳號的總行程數。
觀察週數由首次出現至資料觀察期末計算，至少以一週作分母；這不是完整生命週期的使用頻率。

In [ ]:
rider_total = trips.groupby('RiderId', observed=True).size().rename('rider_rides')
rider_areas = trips.groupby(['RiderId', 'pickup_area'],
    observed=True).size().rename('rides').reset_index().join(rider_total, on='RiderId')
rider_areas['share'] = rider_areas['rides'] / rider_areas['rider_rides']
rider_areas = rider_areas.sort_values(['RiderId', 'rides'], ascending=[True, False])
rider_routes = trips.groupby(['RiderId', 'pickup_area', 'dropoff_area'], observed=True).agg(rides=('Id',
    'size'), active_weeks=('created_week', 'nunique')).reset_index().join(rider_total, on='RiderId')
rider_routes['share'] = rider_routes['rides'] / rider_routes['rider_rides']
rider_routes['repeat_candidate'] = rider_routes['rides'].ge(3) & rider_routes['active_weeks'].ge(2)
rider_routes = rider_routes.sort_values(['RiderId', 'rides'], ascending=[True, False])
best_area = rider_areas.drop_duplicates('RiderId')
best_route = rider_routes.drop_duplicates('RiderId')
print('乘客總數：', len(rider_total))
print('有候選規律路線的乘客：', rider_routes.loc[rider_routes.repeat_candidate, 'RiderId'].nunique())
display(best_area.loc[best_area.rider_rides.ge(3), 'share'].describe(percentiles=[0.25, 0.5, 0.75, 0.9]))
display(best_route.loc[best_route.rider_rides.ge(3), 'share'].describe(percentiles=[0.25, 0.5, 0.75, 0.9]))


In [ ]:
riders = trips.groupby('RiderId', observed=True).agg(rides=('Id', 'size'),
    first_seen=('OrderFormCreatedTime_UTC8', 'min'), last_seen=('OrderFormCreatedTime_UTC8', 'max'),
    active_weeks=('created_week', 'nunique'), distinct_pickup_areas=('pickup_area', 'nunique'))
window_end = trips['OrderFormCreatedTime_UTC8'].max().normalize()
riders['observed_days'] = (window_end - riders['first_seen'].dt.normalize()).dt.days + 1
riders['observed_weeks'] = riders['observed_days'].clip(lower=7) / 7
riders['rides_per_observed_week'] = riders['rides'] / riders['observed_weeks']
riders['recency_days'] = (window_end - riders['last_seen'].dt.normalize()).dt.days
mature_riders = riders.loc[riders['observed_days'].ge(14)].copy()
print(f'首次出現後可觀察至少 14 天：{len(mature_riders):,} / {len(riders):,} 位')
display(riders[['rides', 'observed_days', 'active_weeks', 'recency_days']].describe(percentiles=[0.25, 0.5,
    0.75, 0.9, 0.95, 0.99]))
display(mature_riders['rides_per_observed_week'].quantile([0.1, 0.25, 0.33, 0.5, 0.67, 0.75, 0.9, 0.95,
    0.99]))


距離情境只為描述分組，不改動清理檔。零距離、超過 200 公里、速度旗標，以及缺值或負距離列為待查；其餘小於 2 公里為短程、2 至未滿 10 公里為中程、10 公里以上為長程。
每星期幾的每小時行程另除以觀察期間中該星期幾的日數；熱門路線同時看帳號數與單一帳號占比，避免少數高頻帳號主導解讀。

In [ ]:
distance_m = pd.to_numeric(trips['TravelDistance'], errors='coerce')
qa_distance_cols = [c for c in ('qa_zero_distance', 'qa_distance_over_200km',
    'qa_speed_over_150kmh') if c in trips.columns]

def flagged(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)
    return series.astype('string').str.strip().str.lower().isin(['true', '1', 'yes'])
flagged_distance = pd.Series(False, index=trips.index)
for col in qa_distance_cols:
    flagged_distance |= flagged(trips[col])
flagged_distance |= distance_m.isna() | distance_m.lt(0)
trips['distance_context'] = np.select([flagged_distance, distance_m.lt(2000), distance_m.lt(10000)], ['待檢查',
    '短程：<2 km', '中程：2–10 km'], default='長程：≥10 km')
context_summary = trips.groupby('distance_context', observed=True).agg(rides=('Id', 'size'),
    riders=('RiderId', 'nunique'), median_distance_m=('TravelDistance', 'median')).sort_values('rides',
    ascending=False)
context_summary['share_of_all_rides'] = context_summary['rides'] / len(trips)
context_hours = pd.crosstab(trips['created_hour'], trips['distance_context'], normalize='columns')
display(context_summary, context_hours.round(3))


In [ ]:
ranked_rides = rider_total.sort_values(ascending=False)
head_1pct_n = max(1, int(np.ceil(len(ranked_rides) * 0.01)))
print('最多趟數的帳號：', int(ranked_rides.iloc[0]))
print('前 10 個帳號貢獻比例：', round(ranked_rides.head(10).sum() / len(trips), 4))
print('前 1% 帳號貢獻比例：', round(ranked_rides.head(head_1pct_n).sum() / len(trips), 4))
print('行程數至少 100 趟的帳號數：', int(ranked_rides.ge(100).sum()))
calendar_dates = pd.date_range(trips['OrderFormCreatedTime_UTC8'].min().normalize(),
    trips['OrderFormCreatedTime_UTC8'].max().normalize(), freq='D')
weekday_days = pd.Series(calendar_dates.dayofweek).value_counts().rename('calendar_days')
normalized_hours = hourly.reset_index().join(weekday_days, on='created_weekday')
normalized_hours['rides_per_day'] = normalized_hours['rides'] / normalized_hours['calendar_days']
normalized_hours['weekday_name'] = normalized_hours['created_weekday'].map(dict(enumerate(['週一', '週二', '週三',
    '週四', '週五', '週六', '週日'])))
display(normalized_hours[['weekday_name', 'created_hour', 'rides', 'riders', 'calendar_days',
    'rides_per_day']].sort_values('rides_per_day', ascending=False).head(20))

def largest_account_rides(df, keys):
    leaders = df.head(15).reset_index()[keys]
    selected = trips.merge(leaders, on=keys, how='inner')
    account_counts = selected.groupby(keys + ['RiderId'],
        observed=True).size().rename('account_rides').reset_index()
    return account_counts.groupby(keys,
        observed=True)['account_rides'].max().rename('max_account_rides').reset_index()
area_leaders = pickup_hotspots.head(15).reset_index()
area_leaders = area_leaders.merge(largest_account_rides(pickup_hotspots, ['pickup_area']), on='pickup_area',
    how='left')
area_leaders['top_account_share'] = area_leaders.max_account_rides / area_leaders.rides
display(area_leaders[['pickup_area', 'rides', 'riders', 'top_account_share', 'lat', 'lon']])
display(pickup_hotspots.sort_values('riders', ascending=False)[['rides', 'riders', 'lat',
    'lon']].head(15).reset_index())
route_leaders = popular_routes.head(15).reset_index()
route_leaders = route_leaders.merge(largest_account_rides(popular_routes, ['pickup_area', 'dropoff_area']),
    on=['pickup_area', 'dropoff_area'], how='left')
route_leaders['top_account_share'] = route_leaders.max_account_rides / route_leaders.rides
display(route_leaders[['pickup_area', 'dropoff_area', 'rides', 'riders', 'top_account_share']])
display(popular_routes.sort_values('riders', ascending=False)[['rides', 'riders']].head(15).reset_index())


## 高頻帳號與重疊行程
高頻門檻為全期間至少 100 趟。先以起始時間排序，與同帳號先前最晚結束時間比較；較嚴格版本只用 1–180 分鐘行程、排除時間差旗標，且重疊至少 5 分鐘。
這些條件用於敏感度分析，不據此斷言共用帳號，也不刪除清理資料。此段使用全期間資料，產生的排除名單含未來資訊。

In [ ]:
HIGH_RIDE_THRESHOLD = 100
high_ids = rider_total.index[rider_total.ge(HIGH_RIDE_THRESHOLD)]
repeat_ids = rider_routes.loc[rider_routes['repeat_candidate'], 'RiderId'].unique()
normal_ids = rider_total.index[~rider_total.index.isin(high_ids)]
normal_eligible = rider_total.loc[normal_ids].ge(3).sum()
normal_repeat = pd.Index(repeat_ids).isin(normal_ids).sum()
print(f'高頻帳號數：{len(high_ids):,}（全部帳號的 {len(high_ids) / len(rider_total):.2%}）')
print(f'高頻帳號行程占比：{rider_total.loc[high_ids].sum() / len(trips):.2%}')
print(f'全部 ≥3 趟帳號之重複路線候選率：{len(repeat_ids) / rider_total.ge(3).sum():.2%}')
print(f'排除 ≥100 趟帳號後 ≥3 趟帳號之候選率：{normal_repeat / normal_eligible:.2%}')
print(f'候選規律路線帳號中 ≥100 趟者：{pd.Index(repeat_ids).isin(high_ids).sum():,}')
if 'OrderEndDateTime_UTC8' not in trips.columns:
    raise ValueError('缺少 OrderEndDateTime_UTC8，無法進行時間重疊檢查')
high_trips = trips.loc[trips['RiderId'].isin(high_ids), ['RiderId', 'OrderStartDateTime_UTC8',
    'OrderEndDateTime_UTC8']].copy()
high_trips['OrderEndDateTime_UTC8'] = pd.to_datetime(high_trips['OrderEndDateTime_UTC8'], errors='coerce')
high_trips = high_trips.sort_values(['RiderId', 'OrderStartDateTime_UTC8'])
previous_latest_end = high_trips.groupby('RiderId',
    observed=True)['OrderEndDateTime_UTC8'].transform(lambda s: s.cummax().shift(1))
high_trips['overlaps_previous'] = high_trips['OrderStartDateTime_UTC8'].lt(previous_latest_end)
high_trips['start_day'] = high_trips['OrderStartDateTime_UTC8'].dt.normalize()
max_rides_one_day = high_trips.groupby(['RiderId', 'start_day'], observed=True).size().max()
print('高頻帳號中，有時間重疊紀錄的帳號數：', int(high_trips.groupby('RiderId',
    observed=True)['overlaps_previous'].any().sum()))
print('高頻帳號中，與先前行程重疊的紀錄占比：', f"{high_trips['overlaps_previous'].mean():.2%}")
print('高頻帳號的單一帳號單日最多趟數：', int(max_rides_one_day))


In [ ]:
valid_high = trips.loc[trips['RiderId'].isin(high_ids), ['RiderId', 'OrderStartDateTime_UTC8',
    'OrderEndDateTime_UTC8'] + (['qa_time_gap_over_2min'] if 'qa_time_gap_over_2min' in trips.columns else [])].copy()
valid_high['OrderEndDateTime_UTC8'] = pd.to_datetime(valid_high['OrderEndDateTime_UTC8'], errors='coerce')
elapsed = valid_high['OrderEndDateTime_UTC8'] - valid_high['OrderStartDateTime_UTC8']
valid_length = elapsed.between(pd.Timedelta(minutes=1), pd.Timedelta(minutes=180))
if 'qa_time_gap_over_2min' in valid_high.columns:
    valid_length &= ~flagged(valid_high['qa_time_gap_over_2min'])
valid_high = valid_high.loc[valid_length].sort_values(['RiderId', 'OrderStartDateTime_UTC8']).copy()
valid_prev_end = valid_high.groupby('RiderId',
    observed=True)['OrderEndDateTime_UTC8'].transform(lambda s: s.cummax().shift(1))
valid_high['overlap_5min'] = (valid_prev_end - valid_high['OrderStartDateTime_UTC8']).ge(pd.Timedelta(minutes=5))
all_overlap_ids = pd.Index(high_trips.loc[high_trips['overlaps_previous'], 'RiderId'].unique())
robust_overlap_ids = pd.Index(valid_high.loc[valid_high['overlap_5min'], 'RiderId'].unique())
print('高頻帳號原始時間重疊帳號數：', len(all_overlap_ids))
print('有效時長行程數：', len(valid_high), '；保守重疊行程數：', int(valid_high['overlap_5min'].sum()))
print('保守重疊占有效時長高頻行程：', f"{valid_high['overlap_5min'].mean():.2%}")
print('至少一次保守重疊的高頻帳號數：', len(robust_overlap_ids))
for label, excluded_ids in [('原始重疊帳號', all_overlap_ids), ('保守重疊帳號', robust_overlap_ids)]:
    remaining = rider_total.index[~rider_total.index.isin(excluded_ids)]
    eligible = rider_total.loc[remaining].ge(3).sum()
    candidates = pd.Index(repeat_ids).isin(remaining).sum()
    print(f'移開{label}後：排除 {len(excluded_ids):,} 個帳號；≥3 趟帳號的候選規律路線率 {candidates / eligible:.2%}')


## 目的地規則回測
使用 2026-04-01 前歷史，評估 2026-04-01 至 2026-05-01（不含）每帳號第一趟，且歷史至少 3 趟；另列至少 5 趟的子群。這是有下一趟的帳號之條件式目的地預測，未評估是否叫車。
個人頻率、全體熱門及最近目的地分別排序。Top-1／Top-3 是前一／前三個候選是否包含實際目的地網格，不是精確地址命中率。
750 公尺容差比較「候選目的地的歷史下車座標中位數」與「本次實際下車座標」；最近目的地規則改用該目的地最近一次下車座標。這不是網格邊界向外擴張。
本段排除 `robust_overlap_ids` 的子群沿用全期間名單，僅供探索性敏感度比較；嚴格歷史版本見後段。四月曾用於探索，不是未接觸的最終測試集。

In [ ]:
APRIL_START = pd.Timestamp('2026-04-01')
MAY_START = pd.Timestamp('2026-05-01')
history = trips.loc[trips['OrderFormCreatedTime_UTC8'].lt(APRIL_START)].copy()
april = trips.loc[trips['OrderFormCreatedTime_UTC8'].ge(APRIL_START) & trips['OrderFormCreatedTime_UTC8'].lt(MAY_START)].copy()
april_first = april.sort_values(['RiderId', 'OrderFormCreatedTime_UTC8', 'Id']).drop_duplicates('RiderId')
history_rides = history.groupby('RiderId', observed=True).size().rename('history_rides')
evaluation = april_first[['RiderId', 'dropoff_area', 'DropOffLatitude', 'DropOffLongitude']].copy()
evaluation = evaluation.rename(columns={'dropoff_area': 'actual_area', 'DropOffLatitude': 'actual_lat',
    'DropOffLongitude': 'actual_lon'})
evaluation = evaluation.join(history_rides, on='RiderId')
evaluation = evaluation.loc[evaluation['history_rides'].ge(3)].copy()
print('2–3 月行程數：', len(history), '；4 月行程數：', len(april))
print('4 月有搭車且此前歷史 ≥3 趟的評估帳號：', len(evaluation))
personal_dest = history.groupby(['RiderId', 'dropoff_area'],
    observed=True).agg(history_destination_rides=('Id', 'size'), pred_lat=('DropOffLatitude', 'median'),
    pred_lon=('DropOffLongitude', 'median')).reset_index().sort_values(['RiderId',
    'history_destination_rides', 'dropoff_area'], ascending=[True, False, True])
personal_dest['pred_rank'] = personal_dest.groupby('RiderId', observed=True).cumcount() + 1
personal_top3 = personal_dest.loc[personal_dest['pred_rank'].le(3), ['RiderId', 'dropoff_area', 'pred_lat',
    'pred_lon', 'pred_rank']]
scored = evaluation.merge(personal_top3, on='RiderId', how='left')
scored['exact_hit'] = scored['actual_area'].eq(scored['dropoff_area']).fillna(False)

def haversine_m(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = [np.deg2rad(np.asarray(x, dtype=float)) for x in (lat1, lon1, lat2, lon2)]
    dlat, dlon = (lat2 - lat1, lon2 - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * 6371000 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
scored['within_750m'] = haversine_m(scored['actual_lat'], scored['actual_lon'], scored['pred_lat'],
    scored['pred_lon']) <= 750
global_dest = history.groupby('dropoff_area', observed=True).agg(history_riders=('RiderId', 'nunique'),
    history_rides=('Id', 'size'), pred_lat=('DropOffLatitude', 'median'), pred_lon=('DropOffLongitude',
    'median')).reset_index().sort_values(['history_riders', 'history_rides', 'dropoff_area'],
    ascending=[False, False, True]).head(3).copy()
global_dest['pred_rank'] = range(1, len(global_dest) + 1)
global_scored = evaluation.merge(global_dest, how='cross')
global_scored['exact_hit'] = global_scored['actual_area'].eq(global_scored['dropoff_area'])
global_scored['within_750m'] = haversine_m(global_scored['actual_lat'], global_scored['actual_lon'],
    global_scored['pred_lat'], global_scored['pred_lon']) <= 750
for min_history in (3, 5):
    for cohort_name, exclude in [('所有符合條件帳號', False), ('移開保守重疊帳號', True)]:
        cohort = evaluation.loc[evaluation['history_rides'].ge(min_history), 'RiderId']
        if exclude:
            cohort = cohort.loc[~cohort.isin(robust_overlap_ids)]
        if not len(cohort):
            print(f'歷史 ≥{min_history} 趟／{cohort_name}：無可評估帳號')
            continue
        for baseline_name, predictions in [('個人常去地點', scored), ('全體熱門地點', global_scored)]:
            cohort_predictions = predictions.loc[predictions['RiderId'].isin(cohort)]
            for top_k in (1, 3):
                results = cohort_predictions.loc[cohort_predictions['pred_rank'].le(top_k)].groupby('RiderId', observed=True)[['exact_hit', 'within_750m']].any()
                results = results.reindex(cohort, fill_value=False)
                print(f'歷史 ≥{min_history} 趟／{cohort_name}／{baseline_name}／Top-{top_k}：n={len(cohort):,}，網格命中={results.exact_hit.mean():.2%}，距離 ≤750m={results.within_750m.mean():.2%}')


In [ ]:
april_accounts = april['RiderId'].nunique()
prior_known_accounts = april_first['RiderId'].isin(history_rides.index).sum()
print(f'四月有搭車的帳號：{april_accounts:,}')
print(f'其中二至三月有行程的帳號：{prior_known_accounts:,}（{prior_known_accounts / april_accounts:.2%}）')
print(f'其中二至三月 ≥3 趟的帳號：{len(evaluation):,}（{len(evaluation) / april_accounts:.2%}）')
recent_dest = history.sort_values(['RiderId', 'OrderFormCreatedTime_UTC8', 'Id'], ascending=[True, False,
    False]).drop_duplicates(['RiderId', 'dropoff_area'])[['RiderId', 'dropoff_area', 'DropOffLatitude',
    'DropOffLongitude']].copy()
recent_dest['pred_rank'] = recent_dest.groupby('RiderId', observed=True).cumcount() + 1
recent_top3 = recent_dest.loc[recent_dest['pred_rank'].le(3)].rename(columns={'DropOffLatitude': 'pred_lat',
    'DropOffLongitude': 'pred_lon'})
recent_scored = evaluation.merge(recent_top3, on='RiderId', how='left')
recent_scored['exact_hit'] = recent_scored['actual_area'].eq(recent_scored['dropoff_area']).fillna(False)
recent_scored['within_750m'] = haversine_m(recent_scored['actual_lat'], recent_scored['actual_lon'],
    recent_scored['pred_lat'], recent_scored['pred_lon']) <= 750
frequency_top10 = personal_dest.loc[personal_dest['pred_rank'].le(10), ['RiderId', 'dropoff_area',
    'pred_lat', 'pred_lon', 'pred_rank']]
frequency_scored = evaluation.merge(frequency_top10, on='RiderId', how='left')
frequency_scored['exact_hit'] = frequency_scored['actual_area'].eq(frequency_scored['dropoff_area']).fillna(False)
frequency_scored['within_750m'] = haversine_m(frequency_scored['actual_lat'],
    frequency_scored['actual_lon'], frequency_scored['pred_lat'], frequency_scored['pred_lon']) <= 750
for baseline_name, result_table, candidate_sizes in [('個人最常去', frequency_scored, (1, 3, 5, 10)), ('個人最近去過',
    recent_scored, (1, 3))]:
    for prior_min in (3, 5):
        cohort = evaluation.loc[evaluation['history_rides'].ge(prior_min), 'RiderId']
        subset = result_table.loc[result_table['RiderId'].isin(cohort)]
        for k in candidate_sizes:
            hits = subset.loc[subset['pred_rank'].le(k)].groupby('RiderId', observed=True)[['exact_hit',
                'within_750m']].any().reindex(cohort, fill_value=False)
            print(f'歷史 ≥{prior_min} 趟／{baseline_name} Top-{k}：n={len(cohort):,}，精確網格={hits.exact_hit.mean():.2%}，750m內={hits.within_750m.mean():.2%}')
seen_before = evaluation[['RiderId', 'actual_area']].merge(personal_dest[['RiderId',
    'dropoff_area']].drop_duplicates(), left_on=['RiderId', 'actual_area'], right_on=['RiderId',
    'dropoff_area'], how='left', indicator=True)
print('歷史 ≥3 趟且四月目的地曾出現在個人歷史網格：', f"{seen_before['_merge'].eq('both').mean():.2%}")


## 常用時段與路線距離
常用時段按訂單建立時間的星期幾及固定兩小時區間分組；帳號需至少 5 趟且跨 3 週，候選時段需至少 3 趟且跨 2 週。此段仍為全期間描述。
路線距離以「帳號／上車網格／下車網格」為分組單位，先排除待查距離與低於直線距離旗標，再取有效行程距離中位數。只摘要符合重複路線條件的組合，短中長程界線為 2、10 公里；比例分母是有有效距離中位數的候選路線數。

In [ ]:
rider_summary = riders.copy()
rider_summary['ride_count_band'] = pd.cut(rider_summary['rides'], bins=[0, 1, 2, 4, 9, 19, np.inf],
    labels=['1', '2', '3–4', '5–9', '10–19', '20+'])
frequency_summary = rider_summary.groupby('ride_count_band', observed=True).agg(accounts=('rides', 'size'),
    rides=('rides', 'sum'), median_active_weeks=('active_weeks', 'median'),
    median_observed_days=('observed_days', 'median'), median_recency_days=('recency_days', 'median'))
frequency_summary['account_share'] = frequency_summary['accounts'] / len(rider_summary)
frequency_summary['ride_share'] = frequency_summary['rides'] / len(trips)
display(frequency_summary)
time_slots = trips[['RiderId', 'created_weekday', 'created_hour', 'created_week', 'Id']].copy()
time_slots['two_hour_start'] = time_slots['created_hour'] // 2 * 2
slot_counts = time_slots.groupby(['RiderId', 'created_weekday', 'two_hour_start'],
    observed=True).agg(rides=('Id', 'size'), weeks=('created_week', 'nunique')).reset_index()
slot_accounts = rider_total.index[rider_total.ge(5) & riders['active_weeks'].ge(3)]
regular_slots = slot_counts.loc[slot_counts['rides'].ge(3) & slot_counts['weeks'].ge(2)]
regular_time_accounts = pd.Index(regular_slots['RiderId'].unique())
covered_time_accounts = regular_time_accounts.isin(slot_accounts).sum()
print(f'至少 5 趟且跨至少 3 週帳號：{len(slot_accounts):,}')
print(f'其中有跨週常用時段候選：{covered_time_accounts:,}（{covered_time_accounts / len(slot_accounts):.2%}）')
print('同時有常用時段與重複路線候選的帳號數：',
    int(pd.Index(repeat_ids).isin(regular_time_accounts.intersection(slot_accounts)).sum()))
route_keys = ['RiderId', 'pickup_area', 'dropoff_area']
valid_distance_mask = trips['distance_context'].ne('待檢查')
if 'qa_distance_below_straight_line' in trips.columns:
    valid_distance_mask &= ~flagged(trips['qa_distance_below_straight_line']).fillna(False)
print('本節重複路線距離中位數排除的待檢查或直線距離矛盾行程數：', int((~valid_distance_mask).sum()))
valid_distance_trips = trips.loc[valid_distance_mask, route_keys + ['TravelDistance']]
route_distances = valid_distance_trips.groupby(route_keys,
    observed=True)['TravelDistance'].median().rename('median_distance_m').reset_index()
repeat_routes_with_distance = rider_routes.loc[rider_routes['repeat_candidate'],
    route_keys].merge(route_distances, on=route_keys, how='left')
repeat_routes_with_distance['distance_band'] = pd.cut(repeat_routes_with_distance['median_distance_m'],
    bins=[-np.inf, 2000, 10000, np.inf], right=False, labels=['短程：<2 km', '中程：2–10 km', '長程：≥10 km'])
route_distance_summary = repeat_routes_with_distance.groupby('distance_band',
    observed=True).agg(candidate_routes=('RiderId', 'size'), accounts=('RiderId', 'nunique'))
route_distance_summary['share_of_valid_repeat_routes'] = route_distance_summary['candidate_routes'] / repeat_routes_with_distance['median_distance_m'].notna().sum()
display(route_distance_summary)
print('因行程距離全待檢查而無法歸類的候選路線：', int(repeat_routes_with_distance['median_distance_m'].isna().sum()))


## 僅用歷史資料的敏感度回測與排程
以四月前歷史重新判定高頻與重疊帳號，避免以未來資訊決定回測名單。其餘目的地候選、答案與 Top-k 定義不變。
排程僅用歷史訂單建立時間推導：帳號至少 5 趟／3 週，時段至少 3 趟／2 週；再展開至四月對應星期幾。排程命中率的分母是預排的「帳號／日期／兩小時時段」，分子為該時段至少一筆訂單。另報告行程落在預排時段的比例，其分母是該群四月行程數；兩者都不是人數比例或通知點擊率。
部分比例未處理空子群，解讀結果前需確認分母大於零。

In [ ]:
train_rider_count = history.groupby('RiderId', observed=True).size()
train_high_ids = train_rider_count.index[train_rider_count.ge(HIGH_RIDE_THRESHOLD)]
train_overlap_trips = history.loc[history['RiderId'].isin(train_high_ids), ['RiderId',
    'OrderStartDateTime_UTC8',
    'OrderEndDateTime_UTC8'] + (['qa_time_gap_over_2min'] if 'qa_time_gap_over_2min' in history.columns else [])].copy()
train_overlap_trips['OrderEndDateTime_UTC8'] = pd.to_datetime(train_overlap_trips['OrderEndDateTime_UTC8'],
    errors='coerce')
train_duration = train_overlap_trips['OrderEndDateTime_UTC8'] - train_overlap_trips['OrderStartDateTime_UTC8']
train_valid = train_duration.between(pd.Timedelta(minutes=1), pd.Timedelta(minutes=180))
if 'qa_time_gap_over_2min' in train_overlap_trips:
    train_valid &= ~flagged(train_overlap_trips['qa_time_gap_over_2min']).fillna(False)
train_overlap_trips = train_overlap_trips.loc[train_valid].sort_values(['RiderId',
    'OrderStartDateTime_UTC8'])
train_previous_end = train_overlap_trips.groupby('RiderId',
    observed=True)['OrderEndDateTime_UTC8'].transform(lambda s: s.cummax().shift(1))
train_overlap_trips['overlap_5min'] = (train_previous_end - train_overlap_trips['OrderStartDateTime_UTC8']).ge(pd.Timedelta(minutes=5))
train_only_overlap_ids = pd.Index(train_overlap_trips.loc[train_overlap_trips['overlap_5min'],
    'RiderId'].unique())
print('三月底歷史已達 ≥100 趟的帳號數：', len(train_high_ids))
print('其中三月底前至少一次保守重疊的帳號數：', len(train_only_overlap_ids))
for cohort_name, excluded_ids in [('所有符合條件帳號', pd.Index([])), ('移開三月底已知重疊帳號', train_only_overlap_ids)]:
    cohort = evaluation.loc[~evaluation['RiderId'].isin(excluded_ids), 'RiderId']
    for k in (1, 3):
        hits = scored.loc[scored['RiderId'].isin(cohort) & scored['pred_rank'].le(k)].groupby('RiderId',
            observed=True)['exact_hit'].any().reindex(cohort, fill_value=False)
        print(f'{cohort_name}／個人最常去 Top-{k}：n={len(cohort):,}，網格命中率={hits.mean():.2%}')


In [ ]:
train_rides = history.groupby('RiderId', observed=True).size()
train_weeks = history.groupby('RiderId', observed=True)['created_week'].nunique()
train_slot_eligible = train_rides.index[train_rides.ge(5) & train_weeks.ge(3)]
train_slot_rows = history.loc[history['RiderId'].isin(train_slot_eligible), ['RiderId', 'created_weekday',
    'created_hour', 'created_week', 'Id']].copy()
train_slot_rows['two_hour_start'] = train_slot_rows['created_hour'] // 2 * 2
train_slot_counts = train_slot_rows.groupby(['RiderId', 'created_weekday', 'two_hour_start'],
    observed=True).agg(history_rides=('Id', 'size'), history_weeks=('created_week',
    'nunique')).reset_index()
planned_slots = train_slot_counts.loc[train_slot_counts['history_rides'].ge(3) & train_slot_counts['history_weeks'].ge(2), ['RiderId', 'created_weekday', 'two_hour_start']].copy()
print('三月底有 ≥5 趟且至少跨 3 週帳號：', len(train_slot_eligible))
print('三月底符合時段候選的帳號數：', planned_slots['RiderId'].nunique())
print('符合時段候選的不同帳號／星期／兩小時組合數：', len(planned_slots))
april_calendar = pd.DataFrame({'reminder_date': pd.date_range(APRIL_START, MAY_START - pd.Timedelta(days=1),
    freq='D')})
april_calendar['created_weekday'] = april_calendar['reminder_date'].dt.dayofweek
april_presence = april[['RiderId', 'OrderFormCreatedTime_UTC8']].copy()
april_presence['reminder_date'] = april_presence['OrderFormCreatedTime_UTC8'].dt.normalize()
april_presence['two_hour_start'] = april_presence['OrderFormCreatedTime_UTC8'].dt.hour // 2 * 2
april_presence = april_presence.groupby(['RiderId', 'reminder_date', 'two_hour_start'],
    observed=True).size().rename('orders_in_slot').reset_index()
for label, slots in [('所有三月底符合候選的帳號', planned_slots), ('移開三月底已知重疊帳號',
    planned_slots.loc[~planned_slots['RiderId'].isin(train_only_overlap_ids)])]:
    if slots.empty:
        print(label, '：無可排的提醒時段')
        continue
    reminders = slots.merge(april_calendar, on='created_weekday', how='inner')
    reminders = reminders.merge(april_presence, on=['RiderId', 'reminder_date', 'two_hour_start'],
        how='left')
    reminders['has_trip'] = reminders['orders_in_slot'].fillna(0).gt(0)
    eligible_ids = pd.Index(slots['RiderId'].unique())
    april_for_cohort = april.loc[april['RiderId'].isin(eligible_ids), ['RiderId', 'created_weekday',
        'created_hour', 'Id']].copy()
    april_for_cohort['two_hour_start'] = april_for_cohort['created_hour'] // 2 * 2
    covered = april_for_cohort.merge(slots, on=['RiderId', 'created_weekday', 'two_hour_start'], how='left',
        indicator=True)
    print(f"{label}：帳號={len(eligible_ids):,}，四月排定時段={len(reminders):,}，有搭車的排定時段={reminders.has_trip.sum():,}，排程命中率={reminders.has_trip.mean():.2%}，四月曾搭車帳號={april_for_cohort.RiderId.nunique():,}，四月行程落在事先排定時段的比例={covered['_merge'].eq('both').mean():.2%}")


## 已記錄的回測結果

以下整理自作者既有執行紀錄。本次公開整理核對紀錄文字，未讀取資料集或重新計算結果。只保留彙總命中率，不附帳號數或逐筆答案。

四月首趟目的地回測，對象為三月底前至少 3 趟歷史、且四月有行程的帳號；未排除全期間重疊帳號。

| 規則 | Top-1 網格命中率 | Top-3 網格命中率 |
| --- | ---: | ---: |
| 全體熱門目的地 | 1.96% | 4.29% |
| 個人最近去過目的地 | 23.23% | 43.23% |
| 個人最常去目的地 | 29.76% | 45.77% |

個人最常去規則在 750 公尺目的地容差下，Top-1 為 32.74%、Top-3 為 51.76%；這組距離指標與精確網格命中率不同。僅依三月底歷史辨識並排除重疊帳號後，網格 Top-1／Top-3 為 29.79%／45.81%，為不同子群的敏感度比較。

常用時段的離線排程命中率為 31.58%，分母是預排的帳號／日期／兩小時時段。這不是通知點擊率，亦不代表所有乘客每天會叫車；產品因此採可由使用者確認的卡片，不主張依歷史時段大量自動推播。
